# IMP Demo — Collaborator Guide

Interactive sandbox over `src/`. Use this notebook to try **new data, features, and models** before promoting anything to `main.py`.

## Prerequisites

- Put required parquets under `data/` (see `config/config.py`). Yahoo macros (`MACRO_FEATURES`) download/cache on first aligned-panel build.
- Run cells **top to bottom**. After changing CONFIG or §3 / §5+, re-run from that section through §7.
- If you edit `src/`, restart the kernel (or re-run the import cell) before fitting again.

## Where to edit

| Goal | Edit here | Notes |
| ---- | --------- | ----- |
| Target / split / benchmark | **§0a** `TARGET`, `BENCHMARK_SOURCE`, `FEATURE_COLUMNS`, … | `FEATURE_COLUMNS` is the universe (`enabled_features()` from the four config groups). Benchmark: `yahoo` / `equal` / `cap`. `MARKET_TICKER` may be empty for equal/cap. |
| Per-model feature combos | **§0b** `MODELS` | Like `main.py`: family → model → one or more feature lists. Empty list = skip that model/family. Subset of `FEATURE_COLUMNS` / `enabled_features()`. |
| Built-in lin / tr / nn knobs | **§0c** | Families with any non-empty `MODELS` entry actually run. |
| New daily table (e.g. macro) | **§3** — merge onto `df_align` by `Date` | Market-level: `Date` only. Stock-level: `Date` + `TickerId`. |
| Event / preprocess / metrics | **§4–§7** | Prefer not to change wiring unless you know the pipeline. |
| Custom model (demo-only) | **§5+a / 5+b / 5+c** | Fill `MODELS["custom"]` feature lists. Leave **Wire-up** alone. |

## Feature checklist

1. Edit the four feature groups in `config/config.py` (or override §0a `FEATURE_COLUMNS`); universe for §2–§5.
2. Pick each model's features under **§0b** `MODELS` (must be a subset of the universe).
3. A column that is not on that menu is merged in **§3** (DecisionDate-safe; no lookahead), then added to the matching config group (or §0a list) and the relevant `MODELS` entry.
4. Re-run **§4 → §7** (not only fit).

## Custom model checklist

1. Fill feature lists under **§0b** `MODELS["custom"]` for each custom model id.
2. Edit only **§5+a / 5+b / 5+c** (hyperparams + fitter). Roles mirror lin / tr / nn.
3. Re-run **§5+ → §7**. Do not edit the Wire-up cell unless adding a new model id.

## Demo vs `main.py`

- §3 extra merges and `"custom"` stay in this notebook unless you later move them into `src/` / `config/` / `model/`.
- `FEATURE_COLUMNS` defaults to `enabled_features()`; per-model combos live in `MODELS` (same idea as `main.py`).
- Batch baseline: `python main.py` with the same `MODELS` shape for `lin` / `tr` / `nn` only.


## 0. Config

Shared knobs (`FEATURE_COLUMNS` / `enabled_features()` universe + `MODELS` per-model combos),
then one `LIN` / `TR` / `NN` hyperparameter cell. Non-empty entries in `MODELS`
decide what actually fits.


In [2]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.train as _train
importlib.reload(_train)

from src.report import print_notebook_config
from src.train import resolve_demo_setup


In [3]:
pd.set_option("display.width", None) 
pd.set_option("display.expand_frame_repr", False)  
pd.set_option("display.max_columns", None)

### 0a. Shared


In [4]:
# Shared across families (target / split / benchmark / importance).
# Keep in sync with config/config.py unless you are deliberately A/B testing.
TARGET = "ret_ann_mkt"
# Benchmark for *_mkt labels: "yahoo" | "equal" | "cap"
BENCHMARK_SOURCE = "equal"
# Required for yahoo; may be "" / None when source is equal or cap.
MARKET_TICKER = ""

# Catalog for §2–§5: same as src.features.enabled_features()
# (ORIGINAL → ANALYST → MACRO → ENGINEERED in config/config.py).
# Comment out a name in its config group to disable it globally;
# or assign a custom list here for notebook-only experiments.
from src.features import enabled_features

FEATURE_COLUMNS = enabled_features()

SPLIT_COLUMN = "DecisionDate"
TRAIN_END = "2022-12-31"
VAL_END = "2023-12-31"
IMPUTE_STRATEGY = "median"
RANDOM_STATE = 42


### 0b. Models

Per-model feature combos (same shape as `main.py` `MODELS`). Each model: one
flat list, or a list of lists. Empty = skip that model. Every name must appear
in `FEATURE_COLUMNS` / `enabled_features()` above.


In [ ]:
# Per-model feature combos (synced with main.py). Empty list = skip that model.
MODELS = {
    "lin": {
        "ols": [
            ['log_mcap','vol_21d','vix','dxy','QScoreTrendLong','QScoreSpeedLong'],
        ],
        "ridge": [
            ['log_mcap','vol_21d','vix','dxy','QScoreTrendLong','QScoreSpeedLong'],
        ],
        "lasso": [
            ['par_mkt','vix','dxy','mom_21d','QScoreSpeedLong'],
        ],
    },
    "tr": {
        "decision_tree": [
            ['log_mcap','vol_21d','dxy','QScoreTrendLong'],
        ],
        "random_forest": [
            ['log_mcap','dxy'],
        ],
        "hist_gradient_boosting": [
            ['log_mcap','vol_21d','vix','dxy','QScoreTrendLong'],
        ],
    },
    "nn": {
        "mlp_shallow": [
            ['log_mcap','mom_21d'],
        ],
        "mlp": [
            ['log_mcap','vol_21d','mom_21d'],
        ],
        "mlp_deep": [
            ['par_mkt','log_mcap','vol_21d','QScoreSpeedLong'],
        ],
    },
    "custom": {
        "elasticnet": [],
        "extratrees": [],
        "mlp_linear": [],
    },
}


### 0c. Family hyperparameters

`LIN` / `TR` / `NN` knobs (used when the matching `MODELS` family has non-empty
feature lists). Defaults match `config/config_{lin,tr,nn}.py`.


In [ ]:
LIN = {
    "MODEL_TYPES": ("ols", "ridge", "lasso"),
    "ALPHA": 1.0,
    "ALPHA_GRID": (0.01, 0.1, 1.0, 10.0, 100.0),
    "STANDARDIZE": True,
}

TR = {
    "MODEL_TYPES": ("decision_tree", "random_forest", "hist_gradient_boosting"),
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 4,
    "MAX_DEPTH_GRID": (2, 3, 4, 6, 8),
    "LEARNING_RATE": 0.05,
    "MIN_SAMPLES_LEAF": 20,
    "EARLY_STOPPING_PATIENCE": 10,
    "STANDARDIZE": False,
}

NN = {
    "MODEL_TYPES": ("mlp_shallow", "mlp", "mlp_deep"),
    "HIDDEN_SIZES": {
        "mlp_shallow": (64,),
        "mlp": (64, 32),
        "mlp_deep": (128, 64, 32),
    },
    "ALPHA": 1e-4,
    "ALPHA_GRID": (1e-5, 1e-4, 1e-3, 1e-2),
    "LR": 1e-3,
    "BATCH_SIZE": 64,
    "MAX_ITER": 200,
    "EARLY_STOPPING_PATIENCE": 8,
    "STANDARDIZE": True,
}


### Resolve Active Families And Data Paths

`FAMILIES` is derived from non-empty `MODELS` entries (no separate switch).


In [9]:
def _models_active(model_map: dict) -> bool:
    for value in model_map.values():
        if not value:
            continue
        if all(isinstance(item, str) for item in value):
            return True
        if any(isinstance(item, (list, tuple)) and item for item in value):
            return True
    return False


FAMILIES = tuple(
    name
    for name in ("lin", "tr", "nn", "custom")
    if name in MODELS and _models_active(MODELS[name])
)
if not FAMILIES:
    raise ValueError("MODELS has no non-empty feature lists; nothing to run.")

setup = resolve_demo_setup(FAMILIES, lin=LIN, tr=TR, nn=NN, root=ROOT)
ACTIVE_FAMILIES = list(setup.active_families)
STANDARDIZE = setup.standardize
DATA_DIR = setup.data_dir


In [10]:
print_notebook_config(
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    random_state=RANDOM_STATE,
    standardize=STANDARDIZE,
    active_families=ACTIVE_FAMILIES,
    benchmark_source=BENCHMARK_SOURCE,
    market_ticker=MARKET_TICKER,
    models=MODELS,
)

## 1. Load And Merge

Load the three Parquet tables and build the daily panel `df_all` via `src.prepare` (`load_stock`, `load_earnings`, `load_sentiment`, `merge_panel`).

In [11]:
from src.prepare import load_earnings, load_sentiment, load_stock, merge_panel

In [12]:
stock_data = load_stock(DATA_DIR)
stock_data.head()

          | 0/242507

   TickerId  ShortName InstrumentName       Date  LastPrice    High    Open     Low     Volume  CurrencyId  QScoreTrendLong  QScoreSpeedLong  QScoreLong     MarketCap
0      1256  ABNB:XNGS   AIRBNB INC-A 2024-12-31    131.410  133.19  132.46  131.27  2312564.0         840        -0.857143         0.571429   -0.285714  5.688431e+10
1      1256  ABNB:XNGS   AIRBNB INC-A 2024-12-30    131.810  132.61  131.31  129.85  2458835.0         840        -1.142857         0.428571   -0.714286  5.705746e+10
2      1256  ABNB:XNGS   AIRBNB INC-A 2024-12-27    133.385  134.50  133.59  132.16  2650366.0         840        -1.428571        -0.142857   -1.571429  5.773924e+10
3      1256  ABNB:XNGS   AIRBNB INC-A 2024-12-26    135.320  136.27  134.24  134.24  2067513.0         840        -1.714286        -1.142857   -2.857143  5.857686e+10
4      1256  ABNB:XNGS   AIRBNB INC-A 2024-12-24    134.990  135.07  134.33  132.83  1677060.0         840        -2.000000        -2.285714   -4.285714  5.843401e+1

In [13]:
earnings = load_earnings(DATA_DIR)
earnings.head()

          | 0/3928

         Id  TickerId   EventDate  MarketTimeId
194  736622      1360  2020-01-29           2.0
195  736623      1360  2020-04-29           2.0
196  736624      1360  2020-07-22           2.0
197  736625      1360  2020-10-21           2.0
198  736626      1360  2021-02-03           2.0

In [14]:
sentiment = load_sentiment(DATA_DIR)
sentiment.head()

          | 0/15471

      symbol       date  analystRatingsStrongBuy  analystRatingsBuy  analystRatingsHold  analystRatingsSell  analystRatingsStrongSell ShortName_before  tickerId
3827     DAL 2019-01-01                        7                 11                   3                   0                         0              DAL     15635
12862    TDY 2019-01-01                        1                  3                   3                   0                         0              TDY      4219
5239     EXR 2019-01-01                        3                  5                  10                   1                         0              EXR      2308
6528     GPN 2019-01-01                       10                 16                   9                   1                         0              GPN      2564
7648    INVH 2019-01-01                        6                  9                   2                   0                         0             INVH     11758

In [15]:
df_all = merge_panel(stock_data, earnings, sentiment)
df_all

          | 0/241252

        TickerId  ShortName         InstrumentName       Date  LastPrice       High      Open       Low     Volume  CurrencyId  QScoreTrendLong  QScoreSpeedLong  QScoreLong     MarketCap  Id EventDate  MarketTimeId Symbol Exchange symbol       date  analystRatingsStrongBuy  analystRatingsBuy  analystRatingsHold  analystRatingsSell  analystRatingsStrongSell ShortName_before  tickerId
0           1256  ABNB:XNGS           AIRBNB INC-A 2024-12-31   131.4100  133.19000  132.4600  131.2700  2312564.0         840        -0.857143         0.571429   -0.285714  5.688431e+10 NaN       NaT           NaN   ABNB     XNGS   ABNB 2024-12-01                      2.0                9.0                28.0                 5.0                       3.0             ABNB    1256.0
1           1256  ABNB:XNGS           AIRBNB INC-A 2024-12-30   131.8100  132.61000  131.3100  129.8500  2458835.0         840        -1.142857         0.428571   -0.714286  5.705746e+10 NaN       NaT           NaN   ABNB     XN

In [16]:
df_summary = df_all.describe(include="all").T
df_summary["dtype"] = df_all.dtypes
df_summary

                             count unique        top    freq                           mean                  min                  25%                  50%                  75%                  max                 std           dtype
TickerId                  241252.0    NaN        NaN     NaN                    5697.279048               1256.0               2420.0               3556.0              11666.0              15733.0         4456.975787           int64
ShortName                   241252    198  ANET:XNYS    2516                            NaN                  NaN                  NaN                  NaN                  NaN                  NaN                 NaN  string[python]
InstrumentName              241252    199  POOL CORP    1258                            NaN                  NaN                  NaN                  NaN                  NaN                  NaN                 NaN  string[python]
Date                        241252    NaN        NaN     NaN  2022-0

## 2. Aligned Panel And Benchmark

Build `df_align` with `src.events.build_aligned_panel` (uses `src.sessions` + `src.benchmark`). Pass §0a `BENCHMARK_SOURCE` / `MARKET_TICKER` explicitly so the demo does not inherit `config.config`.


In [17]:
from src.events import build_aligned_panel, build_event_table
from src.report import print_event_report


In [18]:
df_align = build_aligned_panel(
    df_all,
    data_dir=DATA_DIR,
    market_ticker=MARKET_TICKER,
    benchmark_source=BENCHMARK_SOURCE,
)
df_align.head()

          | 0/241252

          | 0/1257

          | 0/1762

   TickerId  ShortName InstrumentName       Date  LastPrice   High    Open       Low      Volume  CurrencyId  QScoreTrendLong  QScoreSpeedLong  QScoreLong     MarketCap  Id EventDate  MarketTimeId Symbol Exchange  analystRatingsStrongBuy  analystRatingsBuy  analystRatingsHold  analystRatingsSell  analystRatingsStrongSell  BenchmarkAdjClose        vix    tnx        dxy  Coverage  BuyRatio  SellRatio  RatingScore  mom_21d  vol_21d   log_mcap  par_mkt
0      1256  ABNB:XNGS   AIRBNB INC-A 2020-12-10     144.71  165.0  146.00  141.2500  70447537.0         840              NaN              NaN         NaN  5.917603e+10 NaN       NaT           NaN   ABNB     XNGS                      NaN                NaN                 NaN                 NaN                       NaN         133.749385  22.520000  0.908  90.790001       NaN       NaN        NaN          NaN      NaN      NaN  24.803782      NaN
1      1256  ABNB:XNGS   AIRBNB INC-A 2020-12-11     139.25  151.5  146.55  135.1000  26980830

## 3. New Customized Data / Features

Yahoo macros (`vix` / `tnx` / `dxy`) are already attached in §2 via `config.config.MACRO_FEATURES` (cached like the yahoo benchmark under `data/macro_daily.*`). To use them in models, include them in `MACRO_FEATURES` (or §0a `FEATURE_COLUMNS`).

Optional: merge any *additional* daily table onto `df_align` by `Date` below.


In [19]:
# Macros are on df_align already (from MACRO_FEATURES). Preview:
df_align[["Date", "vix", "tnx", "dxy"]].drop_duplicates("Date").head()

        Date        vix    tnx        dxy
0 2020-12-10  22.520000  0.908  90.790001
1 2020-12-11  23.309999  0.893  90.980003
2 2020-12-14  24.719999  0.892  90.709999
3 2020-12-15  22.889999  0.923  90.470001
4 2020-12-16  22.500000  0.920  90.449997

In [20]:
# Example: merge an extra local table (not required for vix/tnx/dxy).
""" extra = pd.read_parquet(DATA_DIR / "my_extra_daily.parquet")
df_align = df_align.merge(extra, on="Date", how="left")
df_align.head() """

' extra = pd.read_parquet(DATA_DIR / "my_extra_daily.parquet")\ndf_align = df_align.merge(extra, on="Date", how="left")\ndf_align.head() '

Build decision-day features on `df_align`: `mom_21d`, `vol_21d`, `log_mcap` (no lookahead; known at that close).

In [21]:
""" WINDOW = 21

df_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)
price = df_align.groupby("TickerId", sort=False)["LastPrice"]
ret_1d = price.pct_change()
mcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")

# Calculate the percentage change over the last 21 days 
df_align["mom_21d"] = price.pct_change(WINDOW)

# Calculate the standard deviation of the daily percentage changes over the last 21 days
df_align["vol_21d"] = (
    ret_1d.groupby(df_align["TickerId"], sort=False)
    .rolling(WINDOW, min_periods=WINDOW)
    .std(ddof=1)
    .reset_index(level=0, drop=True)
)

# Calculate the log of market cap
df_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)

df_align.head() """

' WINDOW = 21\n\ndf_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)\nprice = df_align.groupby("TickerId", sort=False)["LastPrice"]\nret_1d = price.pct_change()\nmcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")\n\n# Calculate the percentage change over the last 21 days \ndf_align["mom_21d"] = price.pct_change(WINDOW)\n\n# Calculate the standard deviation of the daily percentage changes over the last 21 days\ndf_align["vol_21d"] = (\n    ret_1d.groupby(df_align["TickerId"], sort=False)\n    .rolling(WINDOW, min_periods=WINDOW)\n    .std(ddof=1)\n    .reset_index(level=0, drop=True)\n)\n\n# Calculate the log of market cap\ndf_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)\n\ndf_align.head() '

## 4. Event Table

Build one-row-per-announcement `df` with `src.events.build_event_table` (reuses `df_align`); print descriptive stats with `src.report.print_event_report`.


In [22]:
df = build_event_table(
    df_all,
    data_dir=DATA_DIR,
    df_align=df_align,
    feature_columns=FEATURE_COLUMNS,
    market_ticker=MARKET_TICKER,
    benchmark_source=BENCHMARK_SOURCE,
)
print_event_report(df, primary_target=TARGET, feature_columns=FEATURE_COLUMNS)


          | 0/3821


------------------------------------------------------------
df coverage
------------------------------------------------------------
rows=3817  cols=36  tickers=199
EventDate        2020-01-14 .. 2024-12-19
DecisionDate     2020-01-10 .. 2024-12-17
MarketTimeId
1    1680
2    2137

------------------------------------------------------------
df head
------------------------------------------------------------
   TickerId Symbol  ShortName   EventDate ReleaseDate  AnchorDate DecisionDate  MarketTimeId  ret_ann  ret_ann_mkt  ret_next  ret_next_mkt  ret_post  ret_post_mkt  dir_ann  vol_ann  vol_post  QScoreTrendLong  QScoreSpeedLong  QScoreLong  analystRatingsStrongBuy  analystRatingsBuy  analystRatingsHold  analystRatingsSell  Coverage  BuyRatio  SellRatio  RatingScore     vix    tnx     dxy  mom_21d  vol_21d  log_mcap  par_mkt      Id
0      1256   ABNB  ABNB:XNGS  2021-02-25  2021-02-25  2021-02-26   2021-02-24             2  -0.0189      -0.0164    0.1334        0.1325   -0.1286    

## 5. Preprocess (Train / Val / Test)

Chronological split and train-only impute/scale via `src.preprocess.run_preprocess`; report with `src.report.print_preprocess_report`.


In [23]:
from src.preprocess import run_preprocess
from src.report import print_preprocess_report

In [24]:
result = run_preprocess(
    df,
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    standardize=STANDARDIZE,
)
print_preprocess_report(result, target=TARGET, split_column=SPLIT_COLUMN)
train_y_mean = float(result.folds["train"].y.to_numpy(dtype=float).mean())



------------------------------------------------------------
folds
------------------------------------------------------------
train n= 2254  2020-01-10 .. 2022-12-29
val   n=  771  2023-01-03 .. 2023-12-15
test  n=  789  2024-01-02 .. 2024-12-17
train_y_mean 0.0047693457632490945

------------------------------------------------------------
target summary
------------------------------------------------------------
target=ret_ann_mkt  events_in=3817  dropped_target=3
       count  unique     min    max   mean    std  missing_n missing_rate
fold                                                                     
train   2254    2242 -0.4608 0.4831 0.0048 0.0718          0        0.00%
val      771     767 -0.3640 0.5825 0.0092 0.0815          0        0.00%
test     789     785 -0.4123 0.7124 0.0080 0.0911          0        0.00%

------------------------------------------------------------
train X head
------------------------------------------------------------
   QScoreTrendLong 

## 5+. Custom Model (demo-only)

Enable by filling `MODELS["custom"]` (§0b). Collaborator blocks map to main-family roles:

| Block | Role (like) | Model id |
| ----- | ----------- | -------- |
| **5+a** | `lin` | `elasticnet` |
| **5+b** | `tr` | `extratrees` |
| **5+c** | `nn` | `mlp_linear` |

Edit only 5+a/b/c. **Wire-up** is infrastructure.


In [25]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor

from src.preprocess import PreprocessResult
from src.train import FitBundle

# Which custom models to run (comment a line to skip that lin/tr/nn slot).
CUSTOM_MODEL_TYPES = (
    # "elasticnet",   
    # "extratrees",   
    # "mlp_linear",   
)
CUSTOM_STANDARDIZE = True

# Filled by 5+a / 5+b / 5+c; do not edit by hand.
_CUSTOM_FITTERS = {}
_CUSTOM_PARAMS = {}


### 5+a. `lin` slot → `elasticnet`

Custom stand-in for the **linear** family. Edit only this block.


In [26]:
""" # --- collaborator edit: lin / elasticnet ---
ELASTICNET = {
    "ALPHA": 1.0,
    "L1_RATIO": 0.5,
}


def fit_elasticnet(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    alpha = float(ELASTICNET["ALPHA"])
    l1_ratio = float(ELASTICNET["L1_RATIO"])
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10_000,
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},
    )


_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet
_CUSTOM_PARAMS.update(ELASTICNET) """


' # --- collaborator edit: lin / elasticnet ---\nELASTICNET = {\n    "ALPHA": 1.0,\n    "L1_RATIO": 0.5,\n}\n\n\ndef fit_elasticnet(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    alpha = float(ELASTICNET["ALPHA"])\n    l1_ratio = float(ELASTICNET["L1_RATIO"])\n    model = ElasticNet(\n        alpha=alpha,\n        l1_ratio=l1_ratio,\n        max_iter=10_000,\n        random_state=RANDOM_STATE,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},\n    )\n\n\n_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet\n_CUSTOM_PARAMS.update(ELASTICNET) '

### 5+b. `tr` slot → `extratrees`

Custom stand-in for the **tree** family. Edit only this block.


In [27]:
""" # --- collaborator edit: tr / extratrees ---
EXTRATREES = {
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 6,
    "MIN_SAMPLES_LEAF": 20,
}


def fit_extratrees(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    model = ExtraTreesRegressor(
        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),
        max_depth=int(EXTRATREES["MAX_DEPTH"]),
        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={
            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),
            "max_depth": int(EXTRATREES["MAX_DEPTH"]),
        },
    )


_CUSTOM_FITTERS["extratrees"] = fit_extratrees
_CUSTOM_PARAMS.update(EXTRATREES) """


' # --- collaborator edit: tr / extratrees ---\nEXTRATREES = {\n    "N_ESTIMATORS": 200,\n    "MAX_DEPTH": 6,\n    "MIN_SAMPLES_LEAF": 20,\n}\n\n\ndef fit_extratrees(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    model = ExtraTreesRegressor(\n        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),\n        max_depth=int(EXTRATREES["MAX_DEPTH"]),\n        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),\n        random_state=RANDOM_STATE,\n        n_jobs=-1,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={\n            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),\n            "max_depth": int(EXTRATREES["MAX_DEPTH"]),\n        },\n    )\n\n\n_CUSTOM_FITTERS["extratrees"] = fit_extratrees\n_CUSTOM_PARAMS.update(EXTRATREES) '

### 5+c. `nn` slot → `mlp_linear`

Custom stand-in for the **neural** family. Edit only this block.
(`activation="identity"` ≈ linear map through a hidden layer.)


In [28]:
""" # --- collaborator edit: nn / mlp_linear ---
MLP_LINEAR = {
    "MLP_HIDDEN": (32,),
    "MLP_MAX_ITER": 300,
}


def fit_mlp_linear(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])
    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="identity",
        solver="adam",
        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"hidden": str(hidden)},
    )


_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear
_CUSTOM_PARAMS.update(MLP_LINEAR) """


' # --- collaborator edit: nn / mlp_linear ---\nMLP_LINEAR = {\n    "MLP_HIDDEN": (32,),\n    "MLP_MAX_ITER": 300,\n}\n\n\ndef fit_mlp_linear(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])\n    model = MLPRegressor(\n        hidden_layer_sizes=hidden,\n        activation="identity",\n        solver="adam",\n        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),\n        random_state=RANDOM_STATE,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={"hidden": str(hidden)},\n    )\n\n\n_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear\n_CUSTOM_PARAMS.update(MLP_LINEAR) '

#### Wire-up

Not a lin/tr/nn model block. Errors if `CUSTOM_MODEL_TYPES` names a model that
was not registered by 5+a/b/c; then builds `fit_custom` / `CUSTOM`.


In [29]:
_unknown = [name for name in CUSTOM_MODEL_TYPES if name not in _CUSTOM_FITTERS]
if _unknown:
    raise ValueError(
        "CUSTOM_MODEL_TYPES names are not registered in §5+a/b/c "
        f"(no fit_* block): {_unknown}. "
        f"Registered: {sorted(_CUSTOM_FITTERS)}."
    )
_unused = sorted(set(_CUSTOM_FITTERS) - set(CUSTOM_MODEL_TYPES))
if _unused:
    print("note: registered but not in CUSTOM_MODEL_TYPES:", _unused)


def fit_custom(model_type: str, result: PreprocessResult) -> FitBundle:
    try:
        fitter = _CUSTOM_FITTERS[model_type]
    except KeyError as exc:
        raise ValueError(
            f"Unknown custom model_type={model_type!r}; "
            f"registered={sorted(_CUSTOM_FITTERS)}."
        ) from exc
    return fitter(result)


CUSTOM = {
    "MODEL_TYPES": tuple(CUSTOM_MODEL_TYPES),
    "STANDARDIZE": bool(CUSTOM_STANDARDIZE),
    **_CUSTOM_PARAMS,
}

ACTIVE_FAMILIES = [
    ("custom", dict(CUSTOM)) if name == "custom" else (name, block)
    for name, block in ACTIVE_FAMILIES
]
if ACTIVE_FAMILIES and ACTIVE_FAMILIES[0][0] == "custom":
    STANDARDIZE = bool(CUSTOM["STANDARDIZE"])

print("fit_custom ready:", CUSTOM["MODEL_TYPES"])
print("registered:", sorted(_CUSTOM_FITTERS))


fit_custom ready: ()
registered: []


## 6. Fit And Predict

For each active family: `lin`/`tr`/`nn` via `apply_demo_block`; `custom` via
`build_demo_custom_spec` + §5+ `fit_custom`. For every non-empty entry in
`MODELS`, re-preprocess on that feature combo and `run_one_model`.


In [30]:
from src.metrics import infer_task, order_summary_table, print_summary_table
from src.pipeline import run_one_model
from src.preprocess import run_preprocess
from src.train import apply_demo_block, build_demo_custom_spec


def _feature_key(columns: list[str]) -> str:
    return ",".join(f"'{name}'" for name in columns)


def _normalize_combos(family: str, model_type: str, value: object) -> list[list[str]]:
    if value is None:
        return []
    if not isinstance(value, (list, tuple)):
        raise ValueError(
            f"MODELS[{family!r}][{model_type!r}] must be a feature list "
            f"or a list of feature lists, got {type(value).__name__}."
        )
    if not value:
        return []
    if all(isinstance(item, str) for item in value):
        return [list(value)]
    combos: list[list[str]] = []
    for index, item in enumerate(value):
        if not isinstance(item, (list, tuple)) or not item:
            raise ValueError(
                f"MODELS[{family!r}][{model_type!r}][{index}] must be a "
                "non-empty feature list."
            )
        combos.append(list(item))
    return combos

In [31]:
task = infer_task(TARGET)
if task != "regression":
    raise NotImplementedError(f"Demo supports regression only; got TARGET={TARGET!r}")

available = set(FEATURE_COLUMNS)
summary_rows = []
fit_rows = []
for family, block in ACTIVE_FAMILIES:
    if family == "custom":
        spec = build_demo_custom_spec(
            block,
            fit_one=fit_custom,
            random_state=RANDOM_STATE,
        )
    else:
        spec = apply_demo_block(
            family,
            block,
            random_state=RANDOM_STATE,
        )

    model_map = MODELS.get(family, {})
    print(
        f"\n### family={family}  standardize={block['STANDARDIZE']}  "
        f"models={list(spec.model_types)}"
    )
    for model_type in spec.model_types:
        combos = _normalize_combos(family, model_type, model_map.get(model_type, []))
        if not combos:
            print(f"  skip {model_type} (empty MODELS[{family!r}][{model_type!r}])")
            continue
        for columns in combos:
            unknown = [name for name in columns if name not in available]
            if unknown:
                raise ValueError(
                    f"MODELS[{family!r}][{model_type!r}] has features not in "
                    f"FEATURE_COLUMNS / enabled_features(): {unknown}."
                )
            result_f = run_preprocess(
                df,
                target=TARGET,
                feature_columns=columns,
                split_column=SPLIT_COLUMN,
                train_end=TRAIN_END,
                val_end=VAL_END,
                impute_strategy=IMPUTE_STRATEGY,
                standardize=bool(block["STANDARDIZE"]),
            )
            train_y_mean_f = float(
                result_f.folds["train"].y.to_numpy(dtype=float).mean()
            )
            row, _predictions, record = run_one_model(
                spec,
                model_type=model_type,
                result=result_f,
                train_y_mean=train_y_mean_f,
                task=task,
            )
            feature_key = _feature_key(columns)
            summary_rows.append(
                {"family": family, **row, "features": feature_key}
            )
            fit_rows.append(
                {"family": family, "features": feature_key, **record}
            )

if not summary_rows:
    raise ValueError("No models ran; fill MODELS feature lists.")

summary = order_summary_table(pd.DataFrame(summary_rows)).reset_index(drop=True)
fits = pd.DataFrame(fit_rows)



### family=lin  standardize=True  models=['ols', 'ridge', 'lasso']

OLS

------------------------------------------------------------
METRICS
------------------------------------------------------------
          n         r2 (oos)        rmse (mean/zero)         mae (mean/zero)      ic rank_ic hit_rate
fold                                                                                                 
train  2254  0.0134 (0.0134)  0.0713 (0.0718/0.0719)  0.0515 (0.0517/0.0516)  0.1157  0.0910   0.5275
val     771  0.0167 (0.0196)  0.0808 (0.0816/0.0820)  0.0578 (0.0582/0.0583)  0.1491  0.1074   0.5512
test    789  0.0008 (0.0021)  0.0910 (0.0911/0.0914)  0.0644 (0.0648/0.0648)  0.0541  0.0679   0.5387

------------------------------------------------------------
FEATURE
------------------------------------------------------------
                 delta_rmse  delta_mae  delta_ic  delta_rank_ic  delta_hit_rate  delta_oos_r2
feature                                                      

## 7. Summary

Compact cross-model comparison on the test fold (`print_summary_table`), with
trailing `features` like `main_metrics.csv`.


In [32]:
print_summary_table(summary)
# print(fits)

family model  n_test oos_r2 rank_ic quintile_ls     ic      r2    mae   rmse hit_rate sign_ls                                                                     features
   lin   ols     789 0.0021  0.0679      0.0102 0.0541  0.0008 0.0644 0.0910   0.5387  0.0055 'QScoreTrendLong','QScoreSpeedLong','log_mcap','vol_21d','RatingScore','dxy'
   lin ridge     789 0.0021  0.0679      0.0102 0.0541  0.0008 0.0644 0.0910   0.5387  0.0055 'QScoreTrendLong','QScoreSpeedLong','log_mcap','vol_21d','RatingScore','dxy'
   lin lasso     789 0.0000     n/a         n/a    n/a -0.0012 0.0648 0.0911   0.5184  0.0080                  'QScoreTrendLong','vol_21d','SellRatio','RatingScore','dxy'
